# Phase-D probe: λ-sweep + K-sweep at the undertrained ckpt (ep0049)**Goal**: characterise the hybrid pipeline's sensitivity to its two mainknobs (λ_prior and K) at the most-improvable Phase-B ckpt (ep0049,standalone 46 % test, hybrid K=10 λ=0.5 test = 54 %, +8 pp lift).This notebook runs **two queues** that were moved off the local Mac(too slow there — ~11 hr per ckpt with 4-parallel contention):**Queue 2 — λ-sweep at ep0049 (K=10 fixed)**:  | λ_prior | output dir |  |---:|---|  | 0.1 | `q2_lambda_sweep/ep0049_lam0p1/` |  | 0.25 | `q2_lambda_sweep/ep0049_lam0p25/` |  | 1.0 | `q2_lambda_sweep/ep0049_lam1p0/` |  | 2.0 | `q2_lambda_sweep/ep0049_lam2p0/` |  (λ=0.5 already covered by Phase-D Q0 local sweep.)**Queue 3 — K-sweep at ep0049 (λ=0.5 fixed)**:  | K | output dir |  |---:|---|  | 5  | `q3_k_sweep/ep0049_K5/` |  | 20 | `q3_k_sweep/ep0049_K20/` |  | 40 | `q3_k_sweep/ep0049_K40/` |  (K=10 already covered by Phase-D Q0 local sweep.)### Wall budget (Colab CPU, 2 vCPU free tier)  - Q2 per run: ~90-140 min (K=10)  - Q3 per run: ~50 min (K=5), ~140 min (K=20), ~280 min (K=40)  - Total (7 runs sequential): ~13-16 hr  - Total across 7 Colab tabs in parallel: ~90-280 min wall (limited by K=40)Idempotency: each (ckpt, K, λ) writes to its own Drive sub-dir; re-runningthe notebook skips any run whose result JSON is already complete.

## 1. CPU info

In [ ]:

!cat /proc/cpuinfo | grep 'model name' | head -1
!nproc
!free -h


## 2. Pin numpy 2.0.2 (required for robosuite 1.5.2)

In [ ]:

import subprocess, sys
try:
    import numpy as np
    print(f"  numpy (pre-pin): {np.__version__}")
    needs_pin = not np.__version__.startswith("2.0")
except ImportError:
    needs_pin = True

if needs_pin:
    print("  installing numpy==2.0.2 (~30 s) ...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "--force-reinstall", "numpy==2.0.2",
    ])
    print("  RESTART RUNTIME and re-run this cell.")
    import os
    os.kill(os.getpid(), 9)
else:
    import torch
    print(f"  torch: {torch.__version__}")


## 3. Mount Drive + define pathsRe-uses the Phase-B Drive folder (`MyDrive/hybrid_traj_dp_phase_b/`)for the ckpt and dataset.  Outputs land in a Phase-D sub-folder.

In [ ]:

from google.colab import drive
import os

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/hybrid_traj_dp_phase_b"
assert os.path.isdir(SAVE_DIR), f"missing Phase-B Drive folder: {SAVE_DIR}"

CKPT_DIR = SAVE_DIR
PROBE_OUT_DIR = f"{SAVE_DIR}/phase_d_lambda_k_probe"   # this notebook writes here
os.makedirs(PROBE_OUT_DIR, exist_ok=True)

REPO_DIR = "/content/HRTG-dp"
DATA_DIR = "/content/can_ph"
DATA_PATH = f"{DATA_DIR}/low_dim_v141.hdf5"

# The ckpt this probe varies knobs around:
CKPT_NAME = "v_theta_dp_tobs2_h16_v3b_ep0049.pt"
CKPT_PATH = f"{CKPT_DIR}/{CKPT_NAME}"
assert os.path.isfile(CKPT_PATH), f"missing ckpt {CKPT_PATH}"
print(f"ckpt: {CKPT_PATH}  ({os.path.getsize(CKPT_PATH) / 1e6:.1f} MB)")
print(f"probe outputs -> {PROBE_OUT_DIR}")


## 4. Clone HRTG-dp at `feat/dp-phase-c` (the post-Phase-D-fix branch)

In [ ]:

import os, subprocess
REPO_URL = "https://github.com/watamelon2/Hybrid-Robot-Trajectory-Generation.git"
BRANCH = "feat/dp-phase-c"

if os.path.isdir(REPO_DIR):
    print(f"[update] {REPO_DIR}")
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "checkout", BRANCH])
    subprocess.check_call(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"])
else:
    subprocess.check_call([
        "git", "clone", "--branch", BRANCH, "--single-branch",
        REPO_URL, REPO_DIR,
    ])
subprocess.check_call(["git", "-C", REPO_DIR, "log", "--oneline", "-3"])

# Verify the Phase-D chunk-boundary fix is present.
print("\n[verify] Phase-D fix:")
subprocess.check_call([
    "bash", "-c",
    f"grep -n 'params.K' {REPO_DIR}/experiments/15_unified_hybrid/unified_loop.py | grep -i '> 0'"
])


## 5. Procure dataset (re-use Phase-B's Drive cache)

In [ ]:

import os, shutil, subprocess, sys

DRIVE_CANDIDATES = [
    f"{SAVE_DIR}/low_dim_v141.hdf5",
    "/content/drive/MyDrive/hybrid_traj_dp_phase3p5/low_dim_v141.hdf5",
    "/content/drive/MyDrive/hybrid_traj_dp_phase2b/low_dim_v141.hdf5",
    "/content/drive/MyDrive/hybrid_traj_dp_phase2/low_dim_v141.hdf5",
]

try:
    import robomimic
    print(f"robomimic: {robomimic.__version__}")
except ImportError:
    print("installing robomimic (~30 s)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "robomimic==0.3.0"])
    import robomimic

os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.isfile(DATA_PATH):
    for cand in DRIVE_CANDIDATES:
        if os.path.isfile(cand) and os.path.getsize(cand) > 1_000_000:
            print(f"using Drive copy at {cand}")
            shutil.copy(cand, DATA_PATH)
            break
assert os.path.isfile(DATA_PATH), f"no dataset at {DATA_PATH}"
print(f"dataset OK ({os.path.getsize(DATA_PATH) / 1e6:.1f} MB)")

from pathlib import Path
pkg_dir = Path(robomimic.__file__).resolve().parent.parent
target_dir = pkg_dir / "datasets" / "can" / "ph"
target_dir.mkdir(parents=True, exist_ok=True)
sym_target = target_dir / "low_dim_v141.hdf5"
if sym_target.is_symlink() or sym_target.exists():
    sym_target.unlink()
sym_target.symlink_to(DATA_PATH)
print(f"symlink -> {sym_target}")


## 6. Install robosuite + mujoco

In [ ]:

import subprocess, sys
try:
    import numpy as np, robosuite
    if np.__version__.startswith("2.0"):
        SKIP = True
        print(f"numpy: {np.__version__}  robosuite: {robosuite.__version__}")
    else:
        SKIP = False
except (ImportError, ValueError):
    SKIP = False

if not SKIP:
    print("installing robosuite + mujoco (~2 min)...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "robosuite==1.5.2", "mujoco==3.8.1",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "--force-reinstall", "numpy==2.0.2",
    ])
    import numpy as np, robosuite
    print(f"numpy: {np.__version__}  robosuite: {robosuite.__version__}")


## 7. Run grid + filter for THIS Colab tabEach tab can be assigned ONE run (parallelize across 7 tabs for maxthroughput) by setting `RUN_FILTER` to a unique tag substring, e.g.`"q2_lam0p1"` for the λ=0.1 cell.

In [ ]:

# Each run: (tag, K, lambda_prior).  Tag is used as the Drive sub-dir name.
RUNS = [
    # Queue 2: lambda-sweep at ep0049 (K=10 fixed)
    ("q2_lam0p1",  10, 0.1),
    ("q2_lam0p25", 10, 0.25),
    ("q2_lam1p0",  10, 1.0),
    ("q2_lam2p0",  10, 2.0),
    # Queue 3: K-sweep at ep0049 (lambda=0.5 fixed)
    ("q3_K5",   5,  0.5),
    ("q3_K20",  20, 0.5),
    ("q3_K40",  40, 0.5),
]

# Empty filter = run them all sequentially in this tab (slow).
# Non-empty = run only the runs whose tag contains this string.
# Example: set RUN_FILTER = "q2_lam0p1" in tab 1, "q3_K40" in tab 2, ...
RUN_FILTER = ""

if RUN_FILTER:
    selected = [r for r in RUNS if RUN_FILTER in r[0]]
else:
    selected = list(RUNS)

assert selected, f"no runs match RUN_FILTER={RUN_FILTER!r}"
print(f"this tab will run {len(selected)} eval(s):")
for tag, K, lam in selected:
    print(f"  {tag:<14}  K={K:<3}  lambda={lam}")


## 8. Hybrid eval loop (THE MAIN CELL)Spawns one `run_can.py` subprocess per (tag, K, λ), with idempotentDrive caching.  Streams just the `[exp15]` rollout lines so the celloutput stays readable.

In [ ]:

import subprocess, sys, os, json, time
from pathlib import Path

# Shared knobs (must match local run_phase_c_ksweep.sh / Phase-D Q0).
N_DEMOS, N_ROLLOUTS = 10, 1
N_TEST_ROLLOUTS = 50
TEST_START_SEED = 100_000
H, MPC_REPLANS, MPC_CHUNK_STEPS = 16, 49, 8
MAX_ENV_STEPS = 400
SIGMA_MAX, SIGMA_MIN, PRIOR_N_STEPS = 0.25, 0.05, 100
N_WORKERS = 2

results = []
for tag, K, lam in selected:
    out_dir = f"{PROBE_OUT_DIR}/{tag}"
    os.makedirs(out_dir, exist_ok=True)
    out_name = f"unified_flow_lam-{lam}_K-{K}_R-{MPC_REPLANS}.json"
    out_path = f"{out_dir}/{out_name}"

    # Idempotency: skip if Drive already has a complete result.
    if os.path.isfile(out_path):
        try:
            d = json.load(open(out_path))
            if int(d.get("n_total", 0)) == (N_DEMOS * N_ROLLOUTS + N_TEST_ROLLOUTS):
                print(f"[skip] {tag}  K={K}  lam={lam}  (Drive cached)")
                print(f"       train: {d.get('train_n_succ', '?')}/{d.get('train_n', '?')}")
                print(f"       test : {d.get('test_n_succ', '?')}/{d.get('test_n', '?')}")
                results.append({"tag": tag, "K": K, "lambda": lam, "result": d})
                continue
        except Exception as e:
            print(f"[warn] {out_path} unparseable ({e}); re-eval")

    print(f"\n{'='*72}\n[eval] {tag}  K={K}  lambda={lam}\n{'='*72}")
    t0 = time.time()
    proc = subprocess.Popen([
        sys.executable, "-u",
        "-m", "experiments.15_unified_hybrid.run_can",
        "--task", "can",
        "--prior", "flow",
        "--prior-checkpoint", CKPT_PATH,
        "--H", str(H),
        "--mpc-replans", str(MPC_REPLANS),
        "--mpc-chunk-steps", str(MPC_CHUNK_STEPS),
        "--n-demos", str(N_DEMOS),
        "--n-rollouts", str(N_ROLLOUTS),
        "--n-test-rollouts", str(N_TEST_ROLLOUTS),
        "--test-start-seed", str(TEST_START_SEED),
        "--max-env-steps", str(MAX_ENV_STEPS),
        "--prior-n-steps", str(PRIOR_N_STEPS),
        "--sigma-max", str(SIGMA_MAX),
        "--sigma-min", str(SIGMA_MIN),
        "--lambda-prior", str(lam),
        "--device", "cpu",
        "--seed", "0",
        "--K", str(K),
        "--n-workers", str(N_WORKERS),
        "--output-dir", out_dir,
    ], cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

    for line in proc.stdout:
        if any(tok in line for tok in ("[exp15]", "error", "Error", "Traceback")):
            print(line.rstrip())
    rc = proc.wait()
    dt = time.time() - t0
    print(f"\n[done] rc={rc}  wall={dt/60:.1f} min")

    if rc == 0 and os.path.isfile(out_path):
        d = json.load(open(out_path))
        print(f"  train: {d.get('train_n_succ', '?')}/{d.get('train_n', '?')}")
        print(f"  test : {d.get('test_n_succ', '?')}/{d.get('test_n', '?')}")
        print(f"  total: {d.get('n_success', '?')}/{d.get('n_total', '?')} = "
              f"{float(d.get('success_rate', 0))*100:.1f} %")
        results.append({"tag": tag, "K": K, "lambda": lam, "result": d})
    elif rc != 0:
        print(f"[ERR] eval failed for {tag}  rc={rc}")

print(f"\n{'='*72}\nthis tab finished: {len(results)} eval(s)\n{'='*72}")


## 9. Aggregate Q2 (λ-sweep) results from DriveReads ALL Q2 results from Drive (across all tabs).  Includes theλ=0.5 anchor point from the local Phase-D Q0 sweep if uploaded.

In [ ]:

import os, json, glob, re

# Q2 anchor (lambda=0.5) from Phase-D Q0 (local sweep).
# Test rate: 27/50 = 54 %.  We hardcode it here.
ANCHOR_LAM_0P5 = {
    "lambda": 0.5, "train_n": 10, "train_n_succ": 6,
    "test_n": 50, "test_n_succ": 27,
    "source": "local Q0 sweep (2026-05-30)",
}

q2_rows = [ANCHOR_LAM_0P5]
for d in sorted(glob.glob(f"{PROBE_OUT_DIR}/q2_lam*/unified_flow_lam-*_K-10_R-*.json")):
    js = json.load(open(d))
    q2_rows.append({
        "lambda": float(js.get("lambda_prior", -1)),
        "train_n": int(js.get("train_n", 0)),
        "train_n_succ": int(js.get("train_n_succ", 0)),
        "test_n": int(js.get("test_n", 0)),
        "test_n_succ": int(js.get("test_n_succ", 0)),
        "source": d,
    })
q2_rows.sort(key=lambda r: r["lambda"])

print(f"{'lambda':>7}  {'train':>10}  {'test':>10}  source")
print("-" * 70)
for r in q2_rows:
    train_p = r["train_n_succ"] / max(1, r["train_n"]) * 100
    test_p  = r["test_n_succ"]  / max(1, r["test_n"])  * 100
    src = os.path.basename(r['source']) if r['source'].startswith("/") else r['source']
    print(f"  {r['lambda']:>5.2f}  {r['train_n_succ']}/{r['train_n']}={train_p:>4.0f}%  "
          f"{r['test_n_succ']}/{r['test_n']}={test_p:>4.0f}%  {src}")


## 10. Aggregate Q3 (K-sweep) results from Drive

In [ ]:

import os, json, glob

# Q3 anchor (K=10) from Phase-D Q0 (local sweep).
# Test rate: 27/50 = 54 % (same as Q2 λ=0.5 anchor, by definition).
ANCHOR_K_10 = {
    "K": 10, "train_n": 10, "train_n_succ": 6,
    "test_n": 50, "test_n_succ": 27,
    "source": "local Q0 sweep (2026-05-30)",
}
# Also include K=0 (DP-only) from previous K-sweep, if available
# (test rate at ep0199 was 80 %, but at ep0049 the standalone test was 46 %).
# We add the standalone ep0049 point as K=0:
ANCHOR_K_0 = {
    "K": 0, "train_n": 10, "train_n_succ": -1,    # train rate unknown
    "test_n": 50, "test_n_succ": 23,                # 46 % of 50 = 23
    "source": "Phase-B standalone (DDPM K=100)",
}

q3_rows = [ANCHOR_K_0, ANCHOR_K_10]
for d in sorted(glob.glob(f"{PROBE_OUT_DIR}/q3_K*/unified_flow_lam-0.5_K-*_R-*.json")):
    js = json.load(open(d))
    q3_rows.append({
        "K": int(js.get("K", -1) or js.get("params", {}).get("K", -1)),
        "train_n": int(js.get("train_n", 0)),
        "train_n_succ": int(js.get("train_n_succ", 0)),
        "test_n": int(js.get("test_n", 0)),
        "test_n_succ": int(js.get("test_n_succ", 0)),
        "source": d,
    })
q3_rows.sort(key=lambda r: r["K"])

print(f"{'K':>3}  {'train':>10}  {'test':>10}  source")
print("-" * 70)
for r in q3_rows:
    if r["train_n_succ"] >= 0:
        train_p = r["train_n_succ"] / max(1, r["train_n"]) * 100
        train_str = f"{r['train_n_succ']}/{r['train_n']}={train_p:>4.0f}%"
    else:
        train_str = "    --    "
    test_p  = r["test_n_succ"]  / max(1, r["test_n"])  * 100
    src = os.path.basename(r['source']) if r['source'].startswith("/") else r['source']
    print(f"  {r['K']:>3d}  {train_str}  {r['test_n_succ']}/{r['test_n']}={test_p:>4.0f}%  {src}")


## 11. Plot λ-sweep and K-sweep curves

In [ ]:

import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Q2 plot: test rate vs lambda
xs2 = [r["lambda"] for r in q2_rows]
ys2 = [r["test_n_succ"] / max(1, r["test_n"]) * 100 for r in q2_rows]
ax1.plot(xs2, ys2, "o-", linewidth=2, markersize=8, color="C0")
ax1.axhline(46, color="gray", ls="--", alpha=0.7, label="standalone DP (46 %)")
ax1.set_xlabel("λ_prior")
ax1.set_ylabel("test success rate (%, 50 fresh seeds)")
ax1.set_title("Q2: λ-sweep at ep0049 (K=10)")
ax1.set_xscale("log")
ax1.grid(True, alpha=0.3)
ax1.legend(loc="lower left")
ax1.set_ylim(0, 100)

# Q3 plot: test rate vs K
xs3 = [r["K"] for r in q3_rows]
ys3 = [r["test_n_succ"] / max(1, r["test_n"]) * 100 for r in q3_rows]
ax2.plot(xs3, ys3, "s-", linewidth=2, markersize=8, color="C1")
ax2.axhline(46, color="gray", ls="--", alpha=0.7, label="standalone DP (46 %)")
ax2.set_xlabel("K (MBD denoise depth)")
ax2.set_ylabel("test success rate (%, 50 fresh seeds)")
ax2.set_title("Q3: K-sweep at ep0049 (λ=0.5)")
ax2.grid(True, alpha=0.3)
ax2.legend(loc="lower right")
ax2.set_ylim(0, 100)

plt.tight_layout()
plt.show()

out_png = f"{PROBE_OUT_DIR}/lambda_k_probe.png"
fig.savefig(out_png, dpi=150, bbox_inches="tight")
print(f"saved plot to {out_png}")


## 12. (optional) Sync results back to localAfter Q2 + Q3 finish on Colab, pull the result JSONs to local repo:```bash# On local Mac:python experiments/08_flow_matching_prior/sync_ckpts_from_drive.py \\    --folder-url 'https://drive.google.com/drive/folders/<PROBE_OUT_DIR_ID>?usp=sharing' \\    --pattern 'unified_flow_lam-' \\    --dest experiments/17_dp_hybrid/results/phase_d_2026_05_30/colab_lambda_k_probe```Then update Phase-D conclusion with the λ-vs-test and K-vs-test plots.